# 05 — GitHub Repo Analysis (Evidence-Grounded)

**Why this notebook:** the production GitHub flow in `backend/app/github_analysis/` collapses each repo into a single LLM call over a file-tree dump. Skills get **hallucinated**: no manifest parsing, no language %, no test/CI signal, no fork distinction, no collaboration signal. Output then gets *string-concatenated* into the gap-analysis prompt at `gap_analysis/router.py:100`.

**This notebook fixes that** with a hybrid pipeline:

```
[deterministic prefetch]
   fetch_repos (rich GraphQL) → enrich_repo_signals (manifests, CI, README, languages%)
        ↓
[LangGraph LLM stages]
   repo_evidence_synth (per-repo, parallel) → cross_repo_aggregator → profile_judge
        ↓
   GitHubProfileInsight (backend-ready pydantic)
```

Every inferred skill carries an `evidence_source` + `evidence_snippet` — the LLM must *cite* a file path or import line, not vibe-guess. Manifest-derived skills are pinned `confidence=high` deterministically before the LLM runs.

**Downstream wins:** the final `GitHubProfileInsight` plugs directly into `gap_analysis` (replacing the string concat at `gap_analysis/router.py:100`) and `deep_researcher` (richer skill grounding → better roadmap milestones).


## 0. Setup

Pull secrets from `../backend/.env` so notebook + backend share one source of truth. Kernel: backend `uv` venv (`backend/.venv`).


In [ ]:
import os, json, re, base64, asyncio
from pathlib import Path
from datetime import datetime, timezone
from typing import List, Dict, Any, Optional, Literal, Tuple
from dotenv import load_dotenv

load_dotenv(Path("../backend/.env").resolve())

import nest_asyncio
nest_asyncio.apply()

assert os.getenv("GROQ_API_KEY"), "Missing GROQ_API_KEY in backend/.env"
# Either Supabase OR a GITHUB_PAT must be present.
HAS_SUPABASE = bool(os.getenv("SUPABASE_URL") and os.getenv("SUPABASE_SERVICE_ROLE_KEY"))
HAS_PAT      = bool(os.getenv("GITHUB_PAT"))
assert HAS_SUPABASE or HAS_PAT, "Need SUPABASE_URL + SUPABASE_SERVICE_ROLE_KEY (and USER_ID), or GITHUB_PAT"
print("supabase:", HAS_SUPABASE, "| pat fallback:", HAS_PAT)


## 1. Auth — pull OAuth token from Supabase, fallback to PAT

Mirrors prod: the backend stores per-user OAuth tokens in `github_tokens` ([router.py:66](../backend/app/github_analysis/router.py)). We read that row for a target `USER_ID` so the notebook reflects exactly what the live pipeline would see. If no row exists (notebook running in a fresh env), we fall back to `GITHUB_PAT`.


In [ ]:
import httpx

USER_ID = os.getenv("USER_ID")  # the Supabase auth.users.id of the test user

def load_github_token() -> tuple[str, str]:
    """Returns (access_token, github_username)."""
    if HAS_SUPABASE and USER_ID:
        try:
            from supabase import create_client
            sb = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_ROLE_KEY"])
            resp = sb.table("github_tokens").select("access_token,github_username").eq("user_id", USER_ID).execute()
            if resp.data:
                row = resp.data[0]
                return row["access_token"], row.get("github_username") or ""
            print("  no github_tokens row for USER_ID — falling back to PAT")
        except Exception as e:
            print(f"  supabase lookup failed: {e} — falling back to PAT")
    pat = os.environ["GITHUB_PAT"]
    # cheap whoami to grab login
    r = httpx.get("https://api.github.com/user", headers={"Authorization": f"Bearer {pat}", "User-Agent": "CareerAtlas-NB"})
    r.raise_for_status()
    return pat, r.json()["login"]

ACCESS_TOKEN, VIEWER_LOGIN = load_github_token()
print(f"authenticated as: {VIEWER_LOGIN}")


## 2. Pydantic schemas — backend-ready

These supersede `RepoAnalysisResult` and `ProfileAnalysisResult` ([service.py:13–21](../backend/app/github_analysis/service.py)). The key idea: **every skill carries citable evidence**. The aggregator can rank by `confidence`, the gap-analysis prompt can quote the snippet, and the user UI can surface "we saw `from torch import nn` in `models/resnet.py`" instead of "you know PyTorch (probably)".


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

EngineeringPractice = Literal[
    "tests_present", "ci_configured", "dockerized", "typed", "linted",
    "documented", "monorepo", "uses_pre_commit"
]
SkillCategory      = Literal["language", "framework", "library", "tool", "concept", "platform"]
EvidenceSource     = Literal["manifest", "ci_config", "import_statement", "topic", "readme", "filetree", "inferred"]
Confidence         = Literal["high", "medium", "low"]
RoleInRepo         = Literal["author", "contributor", "fork_learner"]
MaturityTier       = Literal["beginner", "junior", "mid", "senior"]
CollabProfile      = Literal["solo", "occasional-contributor", "active-collaborator"]


class SkillEvidence(BaseModel):
    skill: str
    category: SkillCategory
    confidence: Confidence
    evidence_source: EvidenceSource
    evidence_snippet: str = Field(description="File path or quoted line proving the skill")
    repo: Optional[str] = Field(default=None, description="repo full_name where evidence was found")


class LanguageStat(BaseModel):
    name: str
    bytes: int
    pct: float


class ManifestDep(BaseModel):
    name: str
    version: Optional[str] = None


class Manifest(BaseModel):
    path: str
    kind: Literal["package.json", "requirements.txt", "pyproject.toml", "Pipfile", "go.mod", "Cargo.toml", "pom.xml"]
    deps: List[ManifestDep] = []


class ReadmeQuality(BaseModel):
    length_chars: int = 0
    has_headings: bool = False
    has_install_section: bool = False
    has_usage_section: bool = False
    has_badges: bool = False
    has_images: bool = False
    score: int = 0  # 0–6


class CiSignal(BaseModel):
    workflows: List[str] = []
    has_tests_dir: bool = False
    has_pytest_cfg: bool = False
    has_jest_cfg: bool = False
    has_coverage_cfg: bool = False
    has_pre_commit: bool = False
    has_dockerfile: bool = False


class EnrichedRepo(BaseModel):
    full_name: str
    owner: str
    name: str
    description: Optional[str] = None
    url: str
    stars: int = 0
    pushed_at: Optional[str] = None
    is_fork: bool = False
    parent_full_name: Optional[str] = None
    is_owner: bool = False
    primary_language: Optional[str] = None
    languages: List[LanguageStat] = []
    topics: List[str] = []
    license: Optional[str] = None
    issues_authored: int = 0
    prs_authored: int = 0
    readme_excerpt: str = ""
    readme_quality: ReadmeQuality = ReadmeQuality()
    manifests: List[Manifest] = []
    ci: CiSignal = CiSignal()
    source_excerpts: Dict[str, str] = {}  # path → snippet
    deterministic_skills: List[SkillEvidence] = []
    fetch_errors: List[str] = []


class RepoEvidence(BaseModel):
    repo_full_name: str
    role_in_repo: RoleInRepo
    project_summary: str
    architecture_notes: str
    engineering_practices: List[EngineeringPractice] = []
    code_quality_observations: List[str] = []
    skills_evidence: List[SkillEvidence] = []
    weaknesses: List[str] = []


class GitHubProfileInsight(BaseModel):
    github_username: str
    headline: str
    overall_summary: str
    strengths: List[str] = []
    growth_areas: List[str] = []
    signature_projects: List[str] = []  # full_names of top-3
    collab_profile: CollabProfile
    engineering_maturity: MaturityTier
    language_distribution: List[LanguageStat] = []
    skills_with_evidence: List[SkillEvidence] = []
    per_repo: List[RepoEvidence] = []
    fetch_failures: List[str] = []


## 3. Rich GraphQL fetch

The production query ([service.py:60–98](../backend/app/github_analysis/service.py)) returns only name/description/stars/pushedAt/primaryLanguage. We expand it to also pull:

- `languages(first:10)` → byte-level distribution (instead of "primaryLanguage" only)
- `repositoryTopics` → self-declared stack
- `isFork`, `parent { nameWithOwner }` → distinguish forks from authored work
- `licenseInfo { spdxId }`
- `defaultBranchRef.target ... history.totalCount` → cheap activity signal
- `pullRequests`, `issues` counts on viewer → collaboration profile

One query, 15 repos. Costs ~1 GraphQL request total.


In [ ]:
GRAPHQL_QUERY = """
query($login: String!) {
  viewer {
    login
    pullRequests(first: 1) { totalCount }
    issues(first: 1) { totalCount }
  }
  user(login: $login) {
    repositories(first: 15, orderBy: {field: PUSHED_AT, direction: DESC}, ownerAffiliations: [OWNER, COLLABORATOR]) {
      nodes {
        name
        nameWithOwner
        url
        description
        stargazerCount
        pushedAt
        isFork
        parent { nameWithOwner }
        owner { login }
        primaryLanguage { name }
        licenseInfo { spdxId }
        repositoryTopics(first: 20) { nodes { topic { name } } }
        languages(first: 10, orderBy: {field: SIZE, direction: DESC}) {
          totalSize
          edges { size node { name } }
        }
        defaultBranchRef {
          name
          target {
            ... on Commit {
              history(first: 1) { totalCount }
            }
          }
        }
      }
    }
  }
}
"""

async def gh_graphql(query: str, variables: dict, token: str) -> dict:
    async with httpx.AsyncClient(timeout=30) as client:
        r = await client.post(
            "https://api.github.com/graphql",
            json={"query": query, "variables": variables},
            headers={"Authorization": f"Bearer {token}", "User-Agent": "CareerAtlas-NB"},
        )
        r.raise_for_status()
        data = r.json()
        if "errors" in data:
            raise RuntimeError(f"GraphQL errors: {data['errors']}")
        return data["data"]

async def fetch_raw_repos() -> Tuple[dict, List[dict]]:
    data = await gh_graphql(GRAPHQL_QUERY, {"login": VIEWER_LOGIN}, ACCESS_TOKEN)
    viewer = data["viewer"]
    nodes = data["user"]["repositories"]["nodes"]
    return viewer, nodes

viewer_info, raw_repo_nodes = asyncio.run(fetch_raw_repos())
print(f"viewer prs_total={viewer_info['pullRequests']['totalCount']} issues_total={viewer_info['issues']['totalCount']}")
print(f"fetched {len(raw_repo_nodes)} repos")
for n in raw_repo_nodes[:5]:
    fork_tag = f"  (fork of {n['parent']['nameWithOwner']})" if n['isFork'] and n.get('parent') else ""
    print(f"  • {n['nameWithOwner']:<40} ★{n['stargazerCount']:<4} {n.get('primaryLanguage',{}).get('name') if n.get('primaryLanguage') else '—'}{fork_tag}")


## 4. REST helpers — file tree + raw content

Mirrors `backend/app/github_analysis/github_api.py:38` (`fetch_repo_file_tree`) and `:24` (`fetch_file_content`). Reused as-is, then layered with throttling and retry on 403 (rate limit / abuse).


In [ ]:
GH_REST = "https://api.github.com"
GH_HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}", "User-Agent": "CareerAtlas-NB",
              "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}

_REST_SEM = asyncio.Semaphore(8)  # cap concurrency, avoid abuse-detection

async def gh_get(client: httpx.AsyncClient, url: str, *, accept: Optional[str] = None) -> httpx.Response:
    headers = dict(GH_HEADERS)
    if accept:
        headers["Accept"] = accept
    async with _REST_SEM:
        for attempt in range(3):
            r = await client.get(url, headers=headers, timeout=30)
            if r.status_code == 403 and "rate limit" in r.text.lower():
                await asyncio.sleep(2 ** attempt)
                continue
            return r
        return r

async def fetch_tree(client: httpx.AsyncClient, owner: str, name: str, branch: str) -> List[str]:
    r = await gh_get(client, f"{GH_REST}/repos/{owner}/{name}/git/trees/{branch}?recursive=1")
    if r.status_code != 200:
        return []
    return [it["path"] for it in r.json().get("tree", []) if it["type"] == "blob"]

async def fetch_file(client: httpx.AsyncClient, owner: str, name: str, path: str, *, max_bytes: int = 6000) -> str:
    r = await gh_get(client, f"{GH_REST}/repos/{owner}/{name}/contents/{path}", accept="application/vnd.github.v3.raw")
    if r.status_code != 200:
        return ""
    txt = r.text
    return txt if len(txt) <= max_bytes else txt[:max_bytes] + "\n…(truncated)"


## 5. Deterministic enrichers (the heart of the improvement)

These run **before** any LLM call and produce hard, factual signals. The LLM only fills the gaps (architecture, behavior, weaknesses) — it does **not** generate the skills list from scratch.

- `parse_package_json` / `parse_requirements` / `parse_pyproject` / `parse_pipfile` / `parse_go_mod` / `parse_cargo_toml` / `parse_pom_xml` — pure-Python parsers. Each emits `SkillEvidence(confidence="high", evidence_source="manifest")` with the manifest path as snippet.
- `detect_ci` — scans file tree for workflow + config files.
- `score_readme` — regex-based readme quality 0–6.
- `pick_source_excerpts` — chooses ≤8 representative source files (largest non-test, one per top-level package).
- `extract_imports_from_excerpts` — cheap regex pass to harvest `from X import Y` / `require('X')` → low-confidence skill evidence.


In [ ]:
# ---- Manifest parsers ----
def parse_package_json(text: str, path: str) -> Manifest:
    deps: List[ManifestDep] = []
    try:
        obj = json.loads(text)
        for section in ("dependencies", "devDependencies", "peerDependencies"):
            for n, v in (obj.get(section) or {}).items():
                deps.append(ManifestDep(name=n, version=str(v)))
    except Exception:
        pass
    return Manifest(path=path, kind="package.json", deps=deps)

def parse_requirements(text: str, path: str) -> Manifest:
    deps: List[ManifestDep] = []
    for line in text.splitlines():
        line = line.split("#", 1)[0].strip()
        if not line or line.startswith("-"):
            continue
        m = re.match(r"^([A-Za-z0-9_.\-]+)\s*([<>=!~]+.*)?$", line)
        if m:
            deps.append(ManifestDep(name=m.group(1), version=(m.group(2) or "").strip() or None))
    return Manifest(path=path, kind="requirements.txt", deps=deps)

def parse_pyproject(text: str, path: str) -> Manifest:
    deps: List[ManifestDep] = []
    # [project] dependencies = [...]  and [tool.poetry.dependencies]
    block = re.search(r"\[project\][^\[]*dependencies\s*=\s*\[(.*?)\]", text, re.S)
    if block:
        for raw in re.findall(r'"([^"]+)"', block.group(1)):
            m = re.match(r"^([A-Za-z0-9_.\-]+)\s*([<>=!~,\s.0-9]+)?", raw)
            if m: deps.append(ManifestDep(name=m.group(1), version=(m.group(2) or "").strip() or None))
    poetry = re.search(r"\[tool\.poetry\.dependencies\](.*?)(\n\[|$)", text, re.S)
    if poetry:
        for ln in poetry.group(1).splitlines():
            m = re.match(r'^([A-Za-z0-9_.\-]+)\s*=\s*"?([^"#]+)?', ln.strip())
            if m and m.group(1).lower() != "python":
                deps.append(ManifestDep(name=m.group(1), version=(m.group(2) or "").strip() or None))
    return Manifest(path=path, kind="pyproject.toml", deps=deps)

def parse_pipfile(text: str, path: str) -> Manifest:
    deps: List[ManifestDep] = []
    sec = re.search(r"\[packages\](.*?)(\n\[|$)", text, re.S)
    if sec:
        for ln in sec.group(1).splitlines():
            m = re.match(r'^([A-Za-z0-9_.\-]+)\s*=\s*"?([^"#]+)?', ln.strip())
            if m: deps.append(ManifestDep(name=m.group(1), version=(m.group(2) or "").strip() or None))
    return Manifest(path=path, kind="Pipfile", deps=deps)

def parse_go_mod(text: str, path: str) -> Manifest:
    deps: List[ManifestDep] = []
    for m in re.finditer(r"^\s*([\w./-]+)\s+v[\w.\-+]+", text, re.M):
        deps.append(ManifestDep(name=m.group(1)))
    return Manifest(path=path, kind="go.mod", deps=deps)

def parse_cargo_toml(text: str, path: str) -> Manifest:
    deps: List[ManifestDep] = []
    sec = re.search(r"\[dependencies\](.*?)(\n\[|$)", text, re.S)
    if sec:
        for ln in sec.group(1).splitlines():
            m = re.match(r'^([A-Za-z0-9_.\-]+)\s*=', ln.strip())
            if m: deps.append(ManifestDep(name=m.group(1)))
    return Manifest(path=path, kind="Cargo.toml", deps=deps)

def parse_pom_xml(text: str, path: str) -> Manifest:
    deps: List[ManifestDep] = []
    for m in re.finditer(r"<artifactId>([^<]+)</artifactId>", text):
        deps.append(ManifestDep(name=m.group(1)))
    return Manifest(path=path, kind="pom.xml", deps=deps)

MANIFEST_DISPATCH = [
    ("package.json",      parse_package_json),
    ("requirements.txt",  parse_requirements),
    ("pyproject.toml",    parse_pyproject),
    ("Pipfile",           parse_pipfile),
    ("go.mod",            parse_go_mod),
    ("Cargo.toml",        parse_cargo_toml),
    ("pom.xml",           parse_pom_xml),
]


In [ ]:
# ---- CI / test signal ----
WORKFLOW_RE   = re.compile(r"^\.github/workflows/.*\.ya?ml$")
TEST_DIR_RE   = re.compile(r"^(tests?|__tests__|spec)/", re.I)

def detect_ci(file_paths: List[str]) -> CiSignal:
    workflows = [p for p in file_paths if WORKFLOW_RE.match(p)]
    return CiSignal(
        workflows=workflows,
        has_tests_dir   = any(TEST_DIR_RE.match(p) for p in file_paths),
        has_pytest_cfg  = any(p.endswith(("pytest.ini","tox.ini")) or p == "pyproject.toml" and "[tool.pytest" in p for p in file_paths) or "pytest.ini" in file_paths or "tox.ini" in file_paths,
        has_jest_cfg    = any(p.startswith("jest.config") for p in file_paths),
        has_coverage_cfg= any(p in (".coveragerc","codecov.yml","codecov.yaml",".codecov.yml") for p in file_paths),
        has_pre_commit  = ".pre-commit-config.yaml" in file_paths or ".pre-commit-config.yml" in file_paths,
        has_dockerfile  = "Dockerfile" in file_paths or "docker-compose.yml" in file_paths,
    )

# ---- README quality ----
README_NAMES = {"README.md","README.rst","README.MD","readme.md","README"}

def score_readme(text: str) -> ReadmeQuality:
    if not text:
        return ReadmeQuality()
    rq = ReadmeQuality(length_chars=len(text))
    rq.has_headings = bool(re.search(r"^#{1,3}\s", text, re.M))
    rq.has_install_section = bool(re.search(r"^#{1,3}\s.*(install|setup|getting started)", text, re.M | re.I))
    rq.has_usage_section   = bool(re.search(r"^#{1,3}\s.*(usage|quick start|example)", text, re.M | re.I))
    rq.has_badges          = "shields.io" in text or "badge" in text.lower()
    rq.has_images          = bool(re.search(r"!\[.*?\]\(", text))
    rq.score = sum([rq.length_chars > 500, rq.has_headings, rq.has_install_section, rq.has_usage_section, rq.has_badges, rq.has_images])
    return rq


In [ ]:
# ---- Source excerpt selection ----
SOURCE_EXT_PRIO = {
    ".py": 9, ".ts": 9, ".tsx": 8, ".js": 7, ".jsx": 6,
    ".go": 8, ".rs": 8, ".java": 7, ".kt": 7, ".cpp": 6, ".c": 5, ".cs": 6,
}
SKIP_PARTS = {"node_modules","venv",".venv","dist","build","vendor",".git","__pycache__","target"}
TEST_HINT  = re.compile(r"(?:^|/)(tests?|__tests__|spec)/|_test\.|\.test\.|\.spec\.", re.I)

def pick_source_excerpt_paths(file_paths: List[str], cap: int = 8) -> List[str]:
    cands: List[Tuple[int,str]] = []
    seen_packages: Dict[str, int] = {}
    for p in file_paths:
        parts = p.split("/")
        if any(x in SKIP_PARTS for x in parts): continue
        if TEST_HINT.search(p): continue
        ext = "." + p.rsplit(".",1)[-1].lower() if "." in p else ""
        if ext not in SOURCE_EXT_PRIO: continue
        top = parts[0] if len(parts) > 1 else "_root"
        # bias toward unseen top-level packages
        pkg_bonus = 3 if seen_packages.get(top, 0) == 0 else -seen_packages.get(top, 0)
        score = SOURCE_EXT_PRIO[ext] + pkg_bonus - min(len(parts), 6)
        seen_packages[top] = seen_packages.get(top, 0) + 1
        cands.append((score, p))
    cands.sort(reverse=True)
    return [p for _, p in cands[:cap]]

# ---- Import harvester (low-confidence skill evidence) ----
PY_IMPORT_RE  = re.compile(r"^\s*(?:from\s+([\w\.]+)\s+import|import\s+([\w\.]+))", re.M)
JS_IMPORT_RE  = re.compile(r'(?:from\s+[\'"]([^\'".]+)[\'"]|require\(\s*[\'"]([^\'".]+)[\'"]\s*\))', re.M)

STDLIB_PY = {"os","sys","re","json","math","typing","dataclasses","pathlib","asyncio","datetime","subprocess","logging","collections","itertools","functools"}

def extract_imports(excerpts: Dict[str,str]) -> List[Tuple[str,str,str]]:
    """Returns list of (skill_name, source_path, snippet)."""
    out: List[Tuple[str,str,str]] = []
    for path, text in excerpts.items():
        if path.endswith((".py",)):
            for m in PY_IMPORT_RE.finditer(text):
                mod = (m.group(1) or m.group(2) or "").split(".")[0]
                if mod and mod not in STDLIB_PY:
                    out.append((mod, path, m.group(0).strip()))
        elif path.endswith((".ts",".tsx",".js",".jsx")):
            for m in JS_IMPORT_RE.finditer(text):
                pkg = m.group(1) or m.group(2) or ""
                if pkg and not pkg.startswith("."):
                    out.append((pkg.split("/")[0], path, m.group(0).strip()))
    return out


## 6. Build `EnrichedRepo` per repo

For each repo: fetch tree → find manifests, README, CI configs → fetch+parse → pick source excerpts → fetch them → harvest imports. All concurrent, capped by `_REST_SEM`.


In [ ]:
def _readme_path(file_paths: List[str]) -> Optional[str]:
    for p in file_paths:
        if "/" not in p and p in README_NAMES:
            return p
    for p in file_paths:
        if p.rsplit("/",1)[-1] in README_NAMES:
            return p
    return None

def _seed_deterministic_skills(repo: EnrichedRepo) -> List[SkillEvidence]:
    seeds: List[SkillEvidence] = []
    # topics → high confidence
    for t in repo.topics:
        seeds.append(SkillEvidence(skill=t, category="concept", confidence="high",
                                   evidence_source="topic", evidence_snippet=f"repo topic: {t}", repo=repo.full_name))
    # primary language + all languages with >5% share
    for ls in repo.languages:
        if ls.pct >= 5.0:
            seeds.append(SkillEvidence(skill=ls.name, category="language", confidence="high",
                                       evidence_source="filetree", evidence_snippet=f"{ls.pct:.1f}% of repo bytes", repo=repo.full_name))
    # manifest deps
    for m in repo.manifests:
        for d in m.deps:
            seeds.append(SkillEvidence(skill=d.name, category="library", confidence="high",
                                       evidence_source="manifest", evidence_snippet=f"{m.path}: {d.name}{('@'+d.version) if d.version else ''}", repo=repo.full_name))
    # CI tools
    if repo.ci.workflows:
        seeds.append(SkillEvidence(skill="GitHub Actions", category="tool", confidence="high",
                                   evidence_source="ci_config", evidence_snippet=", ".join(repo.ci.workflows[:3]), repo=repo.full_name))
    if repo.ci.has_dockerfile:
        seeds.append(SkillEvidence(skill="Docker", category="tool", confidence="high",
                                   evidence_source="ci_config", evidence_snippet="Dockerfile present", repo=repo.full_name))
    if repo.ci.has_pytest_cfg or repo.ci.has_tests_dir:
        seeds.append(SkillEvidence(skill="Testing", category="concept", confidence="medium",
                                   evidence_source="ci_config", evidence_snippet="tests/ or pytest.ini present", repo=repo.full_name))
    return seeds


async def enrich_repo(client: httpx.AsyncClient, node: dict) -> EnrichedRepo:
    owner = node["owner"]["login"]; name = node["name"]; full = node["nameWithOwner"]
    languages = []
    lang_block = node.get("languages") or {}
    total = lang_block.get("totalSize") or 0
    for edge in lang_block.get("edges", []) or []:
        sz = edge["size"]
        languages.append(LanguageStat(name=edge["node"]["name"], bytes=sz,
                                       pct=round(100*sz/total, 2) if total else 0.0))
    topics = [t["topic"]["name"] for t in (node.get("repositoryTopics") or {}).get("nodes", [])]
    branch = (node.get("defaultBranchRef") or {}).get("name") or "main"

    repo = EnrichedRepo(
        full_name=full, owner=owner, name=name, description=node.get("description"),
        url=node["url"], stars=node.get("stargazerCount", 0), pushed_at=node.get("pushedAt"),
        is_fork=node.get("isFork", False),
        parent_full_name=(node.get("parent") or {}).get("nameWithOwner"),
        is_owner=(owner.lower() == VIEWER_LOGIN.lower()),
        primary_language=(node.get("primaryLanguage") or {}).get("name"),
        languages=languages, topics=topics,
        license=(node.get("licenseInfo") or {}).get("spdxId"),
    )

    try:
        tree = await fetch_tree(client, owner, name, branch)
    except Exception as e:
        repo.fetch_errors.append(f"tree: {e}"); tree = []

    # README
    rp = _readme_path(tree)
    if rp:
        try:
            txt = await fetch_file(client, owner, name, rp, max_bytes=8000)
            repo.readme_excerpt = txt[:2000]
            repo.readme_quality = score_readme(txt)
        except Exception as e:
            repo.fetch_errors.append(f"readme: {e}")

    # manifests
    manifest_fetches = []
    for fname, parser in MANIFEST_DISPATCH:
        for p in tree:
            if p.rsplit("/",1)[-1] == fname and p.count("/") <= 2:
                manifest_fetches.append((p, parser))
    async def _fetch_manifest(path, parser):
        try:
            txt = await fetch_file(client, owner, name, path, max_bytes=20000)
            return parser(txt, path) if txt else None
        except Exception as e:
            repo.fetch_errors.append(f"manifest {path}: {e}"); return None
    manifests = await asyncio.gather(*[_fetch_manifest(p, fn) for p, fn in manifest_fetches])
    repo.manifests = [m for m in manifests if m]

    # CI signal
    repo.ci = detect_ci(tree)

    # source excerpts
    excerpt_paths = pick_source_excerpt_paths(tree, cap=8)
    async def _fetch_excerpt(p):
        try:
            return p, await fetch_file(client, owner, name, p, max_bytes=3000)
        except Exception as e:
            repo.fetch_errors.append(f"excerpt {p}: {e}"); return p, ""
    pairs = await asyncio.gather(*[_fetch_excerpt(p) for p in excerpt_paths])
    repo.source_excerpts = {p: t for p, t in pairs if t}

    # seed deterministic skills + import harvest
    repo.deterministic_skills = _seed_deterministic_skills(repo)
    for skill, path, snip in extract_imports(repo.source_excerpts):
        repo.deterministic_skills.append(SkillEvidence(
            skill=skill, category="library", confidence="medium",
            evidence_source="import_statement", evidence_snippet=f"{path}: {snip}", repo=repo.full_name))
    return repo

async def build_enriched_repos(nodes: List[dict]) -> List[EnrichedRepo]:
    async with httpx.AsyncClient() as client:
        return await asyncio.gather(*[enrich_repo(client, n) for n in nodes])

enriched = asyncio.run(build_enriched_repos(raw_repo_nodes))
print(f"enriched {len(enriched)} repos")
for r in enriched[:5]:
    fork_tag = " [FORK]" if r.is_fork else ""
    print(f"  • {r.full_name}{fork_tag}: {len(r.manifests)} manifests, {len(r.ci.workflows)} workflows, readme_score={r.readme_quality.score}, {len(r.deterministic_skills)} det-skills")


## 7. LangGraph — `repo_evidence_synth → cross_repo_aggregator → profile_judge`

Deterministic data is in. Now LLM stages add what code can't infer: architecture patterns, code-quality narrative, behavioral weaknesses, and a synthesized profile-level judgment.

**Key design rule:** the `repo_evidence_synth` prompt receives the deterministic skills as a **pinned list** ("already established — only add NEW skills with evidence"). This prevents the LLM from re-listing the manifest deps with lower confidence.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

def get_llm(temp=0.1):
    return ChatGroq(model=GROQ_MODEL, api_key=os.environ["GROQ_API_KEY"], temperature=temp)


class ProfileState(TypedDict):
    enriched: List[EnrichedRepo]
    per_repo: List[RepoEvidence]
    aggregated_skills: List[SkillEvidence]
    language_distribution: List[LanguageStat]
    insight: Optional[GitHubProfileInsight]
    failures: List[str]


REPO_PROMPT = PromptTemplate.from_template("""You are a senior staff engineer reviewing one repo from a candidate's portfolio.

REPO: {full_name}
DESCRIPTION: {description}
LANGUAGES: {languages}
TOPICS: {topics}
IS_FORK: {is_fork} (parent: {parent})
LICENSE: {license}
CI / TESTS: workflows={workflows} tests_dir={tests_dir} dockerfile={dockerfile} pre_commit={pre_commit}
README QUALITY: score={readme_score}/6 length={readme_len}

README EXCERPT:
{readme_excerpt}

MANIFESTS (deterministic, already extracted — do NOT re-list these as skills):
{manifests}

SOURCE EXCERPTS (file → snippet):
{source_excerpts}

DETERMINISTIC SKILLS ALREADY CAPTURED (pinned, do NOT repeat):
{det_skills}

Your job:
1. Decide role_in_repo: "author" if user owns repo and it's not a fork; "fork_learner" if isFork=true; "contributor" otherwise.
2. Write a 2-sentence project_summary grounded in README + filenames.
3. architecture_notes: one paragraph on patterns observed (layered? event-driven? monorepo? hardcoded? scripts vs library?). Reference specific files.
4. engineering_practices: pick from [tests_present, ci_configured, dockerized, typed, linted, documented, monorepo, uses_pre_commit] — only include what evidence supports.
5. code_quality_observations: 2–4 concrete observations. Quote a file or line — no vague claims.
6. skills_evidence: ONLY skills NOT already in deterministic list. For each, set evidence_source ∈ {{import_statement, readme, inferred}}, confidence accordingly, and put the exact file path or quoted line in evidence_snippet.
7. weaknesses: 1–3 honest critiques (e.g., "no tests for core auth flow", "secrets in committed config").

Be specific. No platitudes.
""")


AGG_PROMPT = PromptTemplate.from_template("""You are aggregating per-repo evidence into a single candidate profile.

PER-REPO EVIDENCE (JSON):
{per_repo_json}

LANGUAGE DISTRIBUTION (across all repos, weighted by bytes):
{lang_dist}

COLLABORATION SIGNAL: prs_total={prs_total} issues_total={issues_total}

Produce a GitHubProfileInsight:
- headline: one resume-ready line (e.g. "Backend-focused Python engineer building LLM agent pipelines with FastAPI + LangGraph")
- overall_summary: 3–4 sentences. Concrete. Reference signature repos.
- strengths: bulleted list — each cites a repo
- growth_areas: bulleted list — each cites a repo or pattern observed
- signature_projects: top 3 repo full_names by depth of work (not by stars). Bias against forks.
- collab_profile: solo / occasional-contributor / active-collaborator (use PR + issue counts + presence of contributor work)
- engineering_maturity: beginner / junior / mid / senior — judged from CI+tests+docs+architecture across repos
- language_distribution: copy the input language distribution as-is
- skills_with_evidence: aggregate skills_evidence across all repos. Dedup by skill name — keep the entry with highest confidence; if tied, prefer evidence_source=manifest > ci_config > import_statement > topic > readme > inferred. Cap at 40 entries, sorted by confidence then category.

Do not invent skills not present in any repo's evidence.
""")


async def node_repo_synth(state: ProfileState) -> ProfileState:
    llm = get_llm().with_structured_output(RepoEvidence)
    failures: List[str] = list(state.get("failures", []))

    async def _one(repo: EnrichedRepo) -> Optional[RepoEvidence]:
        det_skills_str = "\n".join(f"- {s.skill} ({s.evidence_source}, {s.confidence})" for s in repo.deterministic_skills) or "(none)"
        manifests_str  = "\n".join(f"- {m.path}: {', '.join(d.name for d in m.deps[:30])}" for m in repo.manifests) or "(none)"
        src_str = "\n\n".join(f"--- {p} ---\n{t}" for p, t in list(repo.source_excerpts.items())[:6]) or "(none)"
        ci = repo.ci
        prompt_args = dict(
            full_name=repo.full_name, description=repo.description or "—",
            languages=", ".join(f"{l.name} ({l.pct}%)" for l in repo.languages[:5]) or "—",
            topics=", ".join(repo.topics) or "—",
            is_fork=repo.is_fork, parent=repo.parent_full_name or "—",
            license=repo.license or "—",
            workflows=", ".join(ci.workflows) or "—",
            tests_dir=ci.has_tests_dir, dockerfile=ci.has_dockerfile, pre_commit=ci.has_pre_commit,
            readme_score=repo.readme_quality.score, readme_len=repo.readme_quality.length_chars,
            readme_excerpt=repo.readme_excerpt[:1500] or "(no readme)",
            manifests=manifests_str, source_excerpts=src_str[:8000],
            det_skills=det_skills_str,
        )
        try:
            ev: RepoEvidence = await (REPO_PROMPT | llm).ainvoke(prompt_args)
            # merge deterministic skills back in (LLM was told not to repeat them — add them now)
            ev.skills_evidence = repo.deterministic_skills + list(ev.skills_evidence)
            ev.repo_full_name = repo.full_name
            return ev
        except Exception as e:
            failures.append(f"{repo.full_name}: {e}")
            return None

    results = await asyncio.gather(*[_one(r) for r in state["enriched"]])
    return {**state, "per_repo": [r for r in results if r], "failures": failures}


def node_aggregate(state: ProfileState) -> ProfileState:
    # weighted language % across repos
    total_bytes: Dict[str, int] = {}
    grand = 0
    for r in state["enriched"]:
        for l in r.languages:
            total_bytes[l.name] = total_bytes.get(l.name, 0) + l.bytes
            grand += l.bytes
    lang_dist = sorted(
        [LanguageStat(name=k, bytes=v, pct=round(100*v/grand, 2) if grand else 0.0) for k, v in total_bytes.items()],
        key=lambda x: -x.bytes
    )[:12]

    # dedup skills cross-repo
    SOURCE_RANK = {"manifest":5, "ci_config":4, "import_statement":3, "topic":2, "readme":1, "filetree":1, "inferred":0}
    CONF_RANK   = {"high":3, "medium":2, "low":1}
    best: Dict[str, SkillEvidence] = {}
    for ev in state["per_repo"]:
        for s in ev.skills_evidence:
            key = s.skill.lower().strip()
            cur = best.get(key)
            if cur is None or (CONF_RANK[s.confidence], SOURCE_RANK[s.evidence_source]) > (CONF_RANK[cur.confidence], SOURCE_RANK[cur.evidence_source]):
                best[key] = s
    aggregated = sorted(best.values(), key=lambda x: (-CONF_RANK[x.confidence], -SOURCE_RANK[x.evidence_source], x.skill.lower()))[:40]
    return {**state, "aggregated_skills": aggregated, "language_distribution": lang_dist}


async def node_judge(state: ProfileState) -> ProfileState:
    llm = get_llm(temp=0.2).with_structured_output(GitHubProfileInsight)
    per_repo_brief = [
        {
            "repo": ev.repo_full_name, "role": ev.role_in_repo,
            "summary": ev.project_summary, "architecture": ev.architecture_notes,
            "practices": list(ev.engineering_practices),
            "weaknesses": ev.weaknesses[:3],
            "skill_count": len(ev.skills_evidence),
        } for ev in state["per_repo"]
    ]
    args = dict(
        per_repo_json=json.dumps(per_repo_brief, indent=2)[:12000],
        lang_dist=", ".join(f"{l.name} {l.pct}%" for l in state["language_distribution"]),
        prs_total=viewer_info["pullRequests"]["totalCount"],
        issues_total=viewer_info["issues"]["totalCount"],
    )
    try:
        insight: GitHubProfileInsight = await (AGG_PROMPT | llm).ainvoke(args)
    except Exception as e:
        state["failures"].append(f"judge: {e}")
        insight = GitHubProfileInsight(
            github_username=VIEWER_LOGIN, headline="(judge failed)", overall_summary=str(e),
            collab_profile="solo", engineering_maturity="junior",
        )
    # force-fill the fields we already computed deterministically
    insight.github_username = VIEWER_LOGIN
    insight.language_distribution = state["language_distribution"]
    insight.skills_with_evidence = state["aggregated_skills"]
    insight.per_repo = state["per_repo"]
    insight.fetch_failures = state["failures"] + [err for r in state["enriched"] for err in r.fetch_errors]
    return {**state, "insight": insight}


graph = StateGraph(ProfileState)
graph.add_node("repo_synth", node_repo_synth)
graph.add_node("aggregate", node_aggregate)
graph.add_node("judge", node_judge)
graph.set_entry_point("repo_synth")
graph.add_edge("repo_synth", "aggregate")
graph.add_edge("aggregate", "judge")
graph.add_edge("judge", END)
app = graph.compile()


In [ ]:
# render the graph
from IPython.display import Image, display
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("mermaid render skipped:", e)
    print(app.get_graph().draw_ascii())


## 8. Run the pipeline


In [ ]:
initial_state: ProfileState = {
    "enriched": enriched,
    "per_repo": [],
    "aggregated_skills": [],
    "language_distribution": [],
    "insight": None,
    "failures": [],
}

final_state = asyncio.run(app.ainvoke(initial_state))
insight: GitHubProfileInsight = final_state["insight"]
print(f"profile produced: {insight.headline}")
print(f"  maturity={insight.engineering_maturity}  collab={insight.collab_profile}")
print(f"  signature projects: {insight.signature_projects}")
print(f"  skills with evidence: {len(insight.skills_with_evidence)}")
print(f"  failures: {len(insight.fetch_failures)}")


## 9. Validation — sanity checks


In [ ]:
# Check 1: ≥1 manifest-sourced skill per repo (where a manifest existed)
for ev in insight.per_repo:
    repo_obj = next((r for r in enriched if r.full_name == ev.repo_full_name), None)
    if repo_obj and repo_obj.manifests:
        manifest_skills = [s for s in ev.skills_evidence if s.evidence_source == "manifest"]
        assert manifest_skills, f"{ev.repo_full_name} has manifests but no manifest-sourced skills"
print("✓ every repo with a manifest has manifest-sourced SkillEvidence")

# Check 2: language % sums ≈ 100
total_pct = sum(l.pct for l in insight.language_distribution)
print(f"✓ language % total = {total_pct:.1f} (expect ~100)")

# Check 3: forks → fork_learner
for ev in insight.per_repo:
    repo_obj = next((r for r in enriched if r.full_name == ev.repo_full_name), None)
    if repo_obj and repo_obj.is_fork:
        assert ev.role_in_repo == "fork_learner", f"{ev.repo_full_name} is fork but role={ev.role_in_repo}"
print("✓ forks classified as fork_learner")

# Check 4: skills dedup
keys = [s.skill.lower() for s in insight.skills_with_evidence]
assert len(keys) == len(set(keys)), "duplicate skills in aggregated list"
print("✓ aggregated skills are deduplicated")

# Check 5: serializable
serialized = insight.model_dump_json(indent=2)
assert json.loads(serialized)
print(f"✓ GitHubProfileInsight serializes to {len(serialized):,} chars of JSON")


In [ ]:
# persist
out_dir = Path("./out"); out_dir.mkdir(exist_ok=True)
out_path = out_dir / "github_profile_insight.json"
out_path.write_text(insight.model_dump_json(indent=2), encoding="utf-8")
print(f"saved → {out_path}")


In [ ]:
# Markdown render
from IPython.display import Markdown

def render_insight(ins: GitHubProfileInsight) -> str:
    md_parts = [
        f"# GitHub Profile — @{ins.github_username}\n",
        f"**{ins.headline}**\n",
        f"**Maturity:** {ins.engineering_maturity}  •  **Collab:** {ins.collab_profile}\n",
        f"\n{ins.overall_summary}\n",
        "## Signature projects",
    ]
    for fn in ins.signature_projects:
        md_parts.append(f"- `{fn}`")
    md_parts.append("\n## Strengths")
    md_parts += [f"- {s}" for s in ins.strengths]
    md_parts.append("\n## Growth areas")
    md_parts += [f"- {g}" for g in ins.growth_areas]
    md_parts.append("\n## Language distribution")
    md_parts += [f"- {l.name}: {l.pct}%" for l in ins.language_distribution[:8]]
    md_parts.append("\n## Top skills (with citations)")
    for s in ins.skills_with_evidence[:25]:
        md_parts.append(f"- **{s.skill}** _({s.category}, {s.confidence}, {s.evidence_source})_ — `{s.evidence_snippet[:140]}`")
    return "\n".join(md_parts)

display(Markdown(render_insight(insight)))


## 10. Per-repo deep dive (debug view)


In [ ]:
for ev in insight.per_repo[:3]:
    print("=" * 80)
    print(f"{ev.repo_full_name}  [{ev.role_in_repo}]")
    print(f"  summary: {ev.project_summary}")
    print(f"  practices: {', '.join(ev.engineering_practices) or '—'}")
    print(f"  architecture: {ev.architecture_notes[:200]}")
    print(f"  weaknesses:")
    for w in ev.weaknesses: print(f"    • {w}")
    print(f"  top skills:")
    for s in ev.skills_evidence[:6]:
        print(f"    • {s.skill:<30} {s.confidence:<6} {s.evidence_source:<18} {s.evidence_snippet[:80]}")


## 11. Backend porting notes

When ready to land this in production:

### 1. Replace schemas
`backend/app/github_analysis/schemas.py` currently defines flat `AnalysisResponse(summary, coding_behavior, skills: List[str])`. Replace with the pydantic models from cell 2 (or move them to a new `schemas/insight.py`).

### 2. Replace `analyze_selected_repositories`
[`backend/app/github_analysis/service.py:199`](../backend/app/github_analysis/service.py) currently does a single LLM call per repo + one aggregator. Replace with the LangGraph pipeline from this notebook — drop `enrich_repo`, `node_repo_synth`, `node_aggregate`, `node_judge` straight into `service.py`.

### 3. DB migration
The `github_profiles` table stores `analysis_summary` (text), `coding_behavior` (text), `inferred_skills` (text[]). Add:
```sql
ALTER TABLE github_profiles
  ADD COLUMN insight_json   JSONB,         -- full GitHubProfileInsight
  ADD COLUMN engineering_maturity TEXT,
  ADD COLUMN collab_profile TEXT,
  ADD COLUMN language_distribution JSONB;
```
The existing text columns can remain as derived flat views for backwards compatibility (`insight.overall_summary` → `analysis_summary`).

### 4. Wire into gap-analysis
`backend/app/gap_analysis/router.py:100` currently appends a stringified summary into `user_headline`. Replace with a **structured block** consumed by the gap-analysis prompt directly:

```python
github_resp = db_client.table("github_profiles").select("insight_json").eq("user_id", user_id).execute()
if github_resp.data and github_resp.data[0].get("insight_json"):
    insight = GitHubProfileInsight.model_validate(github_resp.data[0]["insight_json"])
    github_block = {
        "skills_with_evidence": [s.model_dump() for s in insight.skills_with_evidence],
        "signature_projects": insight.signature_projects,
        "engineering_maturity": insight.engineering_maturity,
    }
    # pass github_block as a named template variable in the gap-analysis prompt
```

In the gap-analysis prompt template (`backend/app/gap_analysis/service.py:26–52`), add a `{{ github_skills_evidence }}` slot. Now the LLM scoring relevance can see "the user has *cited evidence* of using PyTorch in `models/resnet.py:from torch import nn`" and weight skill familiarity correctly.

### 5. Wire into deep researcher
`backend/app/deep_researcher` planner currently takes a flat gap list. Pass `insight.signature_projects` + `insight.engineering_maturity` so it can calibrate milestone difficulty (skip beginner milestones for a `senior` profile, choose advanced project ideas for someone who already shipped a deep-learning service).
